# Évaluer un système RAG — Qualité, Monitoring & Production

**Objectif :** Mesurer si notre RAG est **fiable**, le surveiller dans le temps, et le préparer pour la **production**  
**Prérequis :** avoir fait le notebook `rag-demo.ipynb`  
**Durée :** 45 min

---

### Pourquoi ce notebook ?

Dans `rag-demo.ipynb`, on a **construit** un RAG. Il répondait bien… sur les quelques questions qu'on a testées à la main.

**Mais en entreprise, la vraie question est :**
> « Comment je PROUVE que mon RAG est fiable ? Sur 100 questions, combien de bonnes réponses ? Et comment je le sais s'il se dégrade dans 3 mois ? »

Un RAG en démo est facile. Un RAG **fiable et maintenu** est un métier. C'est ce qui sépare le débutant du profil recherché.

### Plan
1. Reconstruire le RAG (rappel express)
2. Créer un **jeu de test de référence** (golden dataset)
3. **Partie A** — Évaluer la RECHERCHE (Hit Rate, MRR) — sans LLM
4. **Partie B** — Évaluer la GÉNÉRATION (LLM-as-a-judge, la RAG Triad)
5. **Partie C** — Monitoring & maintenance
6. **Partie D** — Mise en production (checklist)

---

## Le principe : on n'améliore que ce qu'on mesure

Un RAG a **2 étages**, et chacun peut échouer indépendamment :

```
Question
   │
   ▼
┌──────────────────┐
│  1. RECHERCHE    │  ← A-t-on trouvé les BONS documents ?
│  (Retrieval)     │     Si non → le LLM n'a aucune chance.
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  2. GÉNÉRATION   │  ← Le LLM a-t-il bien utilisé les documents ?
│  (Generation)    │     Sans inventer, en répondant à la question.
└────────┬─────────┘
         │
         ▼
      Réponse
```

**La règle d'or du debug RAG :**
- Mauvaise recherche → mauvaise réponse **garantie** (on évalue la recherche EN PREMIER)
- Bonne recherche + mauvaise réponse → problème de prompt ou de LLM

C'est pour ça qu'on évalue les **deux étages séparément**.

---

## Étape 0 — Installation & reconstruction du RAG

On réinstalle et on reconstruit le RAG de `rag-demo.ipynb` (mêmes données voyage).

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters chromadb fastembed python-dotenv pandas
print("Installation terminée ✓")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print("Clé API :", "✅ chargée" if GOOGLE_API_KEY else "❌ manquante — vérifiez .env")

In [ ]:
# Reconstruction du RAG (identique à rag-demo.ipynb)
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1-2. Charger + découper
documents = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader,
                            loader_kwargs={"encoding": "utf-8"}).load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100, separators=["\n\n", "\n", ".", " "]
).split_documents(documents)

# 3-4. Embeddings locaux + Chroma
embeddings = FastEmbedEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 5-6. Chaîne RAG
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", google_api_key=GOOGLE_API_KEY, temperature=0.3)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un agent de voyage expert. Réponds UNIQUEMENT à partir du contexte fourni. "
               "Si l'information n'est pas dans le contexte, dis-le honnêtement. Réponds en français."),
    ("human", "Contexte :\n{context}\n\nQuestion : {question}")
])
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
             | prompt | llm | StrOutputParser())

print(f"✅ RAG reconstruit : {len(documents)} docs → {len(chunks)} chunks")

---

## Étape 1 — Le jeu de test de référence (Golden Dataset)

C'est **la fondation** de toute évaluation. Un ensemble de questions dont on connaît :
- la **source attendue** (quel document devrait répondre)
- une **réponse de référence** (la vérité terrain)

> 💡 **En entreprise :** ce jeu est construit avec les experts métier. On commence petit (20-50 questions) et on l'enrichit avec les vraies questions des utilisateurs. **Sans golden dataset, on ne peut pas évaluer.**

Ici on en crée un petit à la main sur nos 3 destinations.

In [ ]:
# Golden dataset : question → source attendue + réponse de référence
golden_dataset = [
    {
        "question": "Quel budget par jour prévoir pour voyager au Japon ?",
        "source_attendue": "japon.txt",
        "reponse_ref": "Entre 80 et 120 euros par jour.",
    },
    {
        "question": "Quand peut-on voir les aurores boréales en Islande ?",
        "source_attendue": "islande.txt",
        "reponse_ref": "De septembre à mars.",
    },
    {
        "question": "Combien coûte l'entrée au Blue Lagoon ?",
        "source_attendue": "islande.txt",
        "reponse_ref": "Environ 75 euros.",
    },
    {
        "question": "Quelle est la ville la plus peuplée du Maroc ?",
        "source_attendue": "maroc.txt",
        "reponse_ref": "Casablanca, avec environ 3,7 millions d'habitants.",
    },
    {
        "question": "Quelle est la hauteur du Mont Fuji ?",
        "source_attendue": "japon.txt",
        "reponse_ref": "3 776 mètres.",
    },
    {
        "question": "Pourquoi Chefchaouen est-elle appelée la ville bleue ?",
        "source_attendue": "maroc.txt",
        "reponse_ref": "Parce que ses ruelles sont peintes en bleu et blanc.",
    },
]

print(f"✅ Golden dataset : {len(golden_dataset)} questions de référence")
for i, item in enumerate(golden_dataset, 1):
    print(f"   {i}. [{item['source_attendue']:11}] {item['question']}")

---

## Partie A — Évaluer la RECHERCHE (Retrieval)

**Sans LLM, sans clé API, gratuit et instantané.** On vérifie si le retriever retrouve les bons documents. Deux métriques standards :

### 📏 Hit Rate @ k (taux de réussite)
> Parmi les **k** chunks retournés, le bon document est-il présent ?

`Hit Rate = nb de questions où la bonne source est trouvée / nb total de questions`

Exemple : Hit Rate @ 3 = 0.90 → dans 90% des cas, le bon doc est dans le top 3.

### 📏 MRR — Mean Reciprocal Rank (rang moyen inversé)
> Non seulement le bon doc est trouvé, mais **à quelle position** ? Plus il est haut, mieux c'est.

`Reciprocal Rank = 1 / position du premier bon chunk`
- Bon chunk en position 1 → score 1.0
- Position 2 → 0.5
- Position 3 → 0.33
- Absent → 0.0

`MRR = moyenne des Reciprocal Ranks sur toutes les questions`

> 💡 **Pourquoi deux métriques ?** Le Hit Rate dit « trouvé ou pas ». Le MRR dit « trouvé, mais bien classé ? ». Un bon retriever a un Hit Rate ET un MRR proches de 1.

In [ ]:
import os
import pandas as pd

def source_name(doc):
    """Récupère juste le nom du fichier depuis les métadonnées."""
    return os.path.basename(doc.metadata.get("source", ""))

def evaluer_retrieval(dataset, retriever, k=3):
    lignes = []
    for item in dataset:
        docs = retriever.invoke(item["question"])
        sources = [source_name(d) for d in docs]

        # Rang du premier chunk provenant de la bonne source (1-indexé)
        rang = next((i + 1 for i, s in enumerate(sources) if s == item["source_attendue"]), None)

        hit = rang is not None
        reciprocal_rank = 1 / rang if rang else 0.0

        lignes.append({
            "question": item["question"][:45] + "…",
            "attendu": item["source_attendue"],
            "trouvé (top k)": ", ".join(sources),
            "hit": "✅" if hit else "❌",
            "rang": rang if rang else "—",
            "RR": round(reciprocal_rank, 2),
        })
    return pd.DataFrame(lignes)

df_retrieval = evaluer_retrieval(golden_dataset, retriever, k=3)
df_retrieval

In [ ]:
# Les scores globaux
hit_rate = (df_retrieval["hit"] == "✅").mean()
mrr = df_retrieval["RR"].mean()

print("📊 SCORES DE RECHERCHE (Retrieval)")
print(f"   Hit Rate @ 3 : {hit_rate:.0%}   (le bon doc est-il dans le top 3 ?)")
print(f"   MRR          : {mrr:.2f}   (est-il bien classé ?)")
print()
if hit_rate == 1.0 and mrr >= 0.9:
    print("✅ Excellent retriever : trouve toujours le bon doc, et en tête.")
elif hit_rate >= 0.8:
    print("🟡 Correct, mais des questions ratent le bon doc — à améliorer.")
else:
    print("🔴 Retriever faible : revoir le chunking ou le modèle d'embeddings.")

---

## Partie B — Évaluer la GÉNÉRATION (LLM-as-a-judge)

La recherche est bonne. Mais la **réponse finale** est-elle bonne ? Problème : évaluer du texte libre à la main sur 100 questions est impossible.

**Solution moderne : le LLM-as-a-judge.** On utilise un LLM comme **correcteur automatique** : on lui donne la question, le contexte et la réponse, et il note.

### La RAG Triad — les 3 questions qui détectent 90% des problèmes

```
                 QUESTION
                /         \
               /           \
   ① Context Relevance    ③ Answer Relevance
      Le contexte est-il     La réponse répond-elle
      pertinent pour          vraiment à la question ?
      la question ?              \
               \                  \
                \                  \
              CONTEXTE ──────────► RÉPONSE
                    ② Groundedness (Fidélité)
                    La réponse est-elle FONDÉE sur
                    le contexte, sans rien inventer ?
```

| Métrique | Question posée | Détecte |
|---|---|---|
| **① Context Relevance** | Les chunks trouvés sont-ils pertinents ? | Mauvaise recherche |
| **② Groundedness** (fidélité) | La réponse s'appuie-t-elle sur le contexte ? | **Hallucination** |
| **③ Answer Relevance** | La réponse adresse-t-elle la question ? | Réponse hors-sujet |

> 💡 **La plus importante = Groundedness.** C'est elle qui attrape les **hallucinations** : quand le LLM invente une info absente du contexte. En entreprise, une hallucination = une perte de confiance.

Ci-dessous, on construit un juge qui note **Groundedness** et **Answer Relevance** de 1 à 5.

In [ ]:
import json
import re

# Le prompt du JUGE : il note la réponse sur 2 critères, en JSON
juge_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es un évaluateur RIGOUREUX de systèmes RAG. On te donne une QUESTION, "
     "le CONTEXTE fourni au modèle, et la RÉPONSE générée. "
     "Tu notes deux critères de 1 (très mauvais) à 5 (excellent) :\n"
     "- groundedness : la réponse est-elle entièrement justifiée par le CONTEXTE ? "
     "(5 = tout est dans le contexte ; 1 = la réponse invente des infos absentes)\n"
     "- pertinence : la réponse répond-elle bien à la QUESTION ? "
     "(5 = répond pleinement ; 1 = hors-sujet)\n"
     "Réponds UNIQUEMENT en JSON valide, sans texte autour, au format exact :\n"
     '{{"groundedness": <int>, "pertinence": <int>, "explication": "<courte phrase>"}}'),
    ("human",
     "QUESTION :\n{question}\n\nCONTEXTE :\n{context}\n\nRÉPONSE À ÉVALUER :\n{reponse}")
])

juge_chain = juge_prompt | llm | StrOutputParser()

def parser_json(texte):
    """Extrait le JSON même si le LLM ajoute des ```json ... ``` autour."""
    match = re.search(r"\{.*\}", texte, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return {"groundedness": None, "pertinence": None, "explication": "parse_error"}

# Test rapide du juge sur un cas
_test = juge_chain.invoke({
    "question": "Quelle est la capitale du Japon ?",
    "context": "Tokyo est la capitale du Japon.",
    "reponse": "La capitale du Japon est Tokyo."
})
print("Test du juge (réponse brute) :", _test[:120])
print("Test du juge (parsé)        :", parser_json(_test))

In [ ]:
import time

# On évalue un SOUS-ENSEMBLE pour économiser le quota Gemini (free tier).
# En production, on évalue TOUT le golden dataset.
sous_ensemble = golden_dataset[:3]

resultats_gen = []
for item in sous_ensemble:
    question = item["question"]

    # 1. Récupérer le contexte ET la réponse du RAG
    docs = retriever.invoke(question)
    contexte = format_docs(docs)
    reponse = rag_chain.invoke(question)

    # 2. Le juge note la réponse
    note = parser_json(juge_chain.invoke({
        "question": question, "context": contexte, "reponse": reponse
    }))

    resultats_gen.append({
        "question": question[:40] + "…",
        "réponse": reponse[:55] + "…",
        "grounded (1-5)": note.get("groundedness"),
        "pertinence (1-5)": note.get("pertinence"),
    })
    time.sleep(4)  # petite pause pour ménager le quota

df_gen = pd.DataFrame(resultats_gen)
df_gen

In [ ]:
# Scores moyens de génération
grounded_moy = df_gen["grounded (1-5)"].dropna().mean()
pertinence_moy = df_gen["pertinence (1-5)"].dropna().mean()

print("📊 SCORES DE GÉNÉRATION (LLM-as-a-judge)")
print(f"   Groundedness moyenne : {grounded_moy:.1f} / 5   (fidélité au contexte, anti-hallucination)")
print(f"   Pertinence moyenne   : {pertinence_moy:.1f} / 5   (répond à la question)")
print()
print("👉 Groundedness élevée = le RAG ne hallucine pas.")
print("👉 Pertinence élevée   = le RAG répond bien aux questions.")

### 🎯 Test clé : le juge attrape-t-il une hallucination ?

On fabrique volontairement une **fausse réponse** (une info absente du contexte) et on vérifie que le juge lui met une **mauvaise note de groundedness**.

In [ ]:
# Contexte réel sur le Japon, mais réponse INVENTÉE (hallucination)
contexte_japon = format_docs(retriever.invoke("budget Japon"))

reponse_hallucinee = "Le budget pour le Japon est de 500 euros par jour et l'entrée au Mont Fuji coûte 200 euros."

note_hallu = parser_json(juge_chain.invoke({
    "question": "Quel budget pour le Japon ?",
    "context": contexte_japon,
    "reponse": reponse_hallucinee
}))

print("Réponse hallucinée testée :")
print(f"  « {reponse_hallucinee} »\n")
print(f"  → Groundedness : {note_hallu.get('groundedness')}/5")
print(f"  → Explication  : {note_hallu.get('explication')}")
print()
if (note_hallu.get("groundedness") or 5) <= 2:
    print("✅ Le juge a bien DÉTECTÉ l'hallucination (note basse) !")
else:
    print("⚠️ Le juge n'a pas détecté l'hallucination — à surveiller.")

---

## Partie C — Monitoring & maintenance

L'évaluation ci-dessus, c'est **avant** la mise en ligne (offline). Une fois en production, un RAG **vit** : les données changent, les utilisateurs posent des questions imprévues, la qualité peut se dégrader.

### Ce qu'il faut LOGGER à chaque requête

| Donnée loggée | Pourquoi |
|---|---|
| Question de l'utilisateur | Comprendre les vrais besoins, enrichir le golden dataset |
| Sources retournées | Débugger « pourquoi cette réponse ? » |
| Réponse générée | Traçabilité, audit |
| Latence (ms) | Détecter les ralentissements |
| Coût (tokens) | Suivre le budget |
| 👍/👎 utilisateur | **Le signal le plus précieux** : la vraie satisfaction |

### La dérive (drift) — l'ennemi silencieux

```
Mois 1 : RAG déployé, Hit Rate 95% ✅
Mois 3 : nouveaux produits ajoutés, mais docs pas mis à jour
         → questions sur les nouveautés → RAG ne trouve rien ❌
Mois 3 : Hit Rate réel tombé à 70%… sans alerte si on ne mesure pas !
```

**Types de dérive :**
- **Data drift** : les documents deviennent obsolètes (prix, produits, règles changent)
- **Query drift** : les utilisateurs posent de nouveaux types de questions
- **Model drift** : le fournisseur met à jour son LLM, le comportement change

### La boucle d'amélioration continue

```
   Production
       │
       ▼
   Logger tout (questions, réponses, 👍/👎)
       │
       ▼
   Repérer les échecs (👎, groundedness basse)
       │
       ▼
   Ajouter ces cas au golden dataset
       │
       ▼
   Ré-évaluer + corriger (chunking, prompt, docs)
       │
       └──────► Redéployer → (recommencer)
```

Ci-dessous, un logger minimal (en vrai on utiliserait une base de données ou un outil comme **LangSmith**).

In [ ]:
import time
import json

def rag_avec_logging(question, fichier_log="rag_logs.jsonl"):
    """Répond à une question ET logge tout dans un fichier (1 ligne JSON par requête)."""
    t0 = time.time()

    docs = retriever.invoke(question)
    sources = [source_name(d) for d in docs]
    reponse = rag_chain.invoke(question)

    latence_ms = int((time.time() - t0) * 1000)

    log = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "question": question,
        "sources": sources,
        "reponse": reponse,
        "latence_ms": latence_ms,
        "feedback": None,   # rempli plus tard par 👍/👎 de l'utilisateur
    }

    # On ajoute une ligne au fichier de logs
    with open(fichier_log, "a", encoding="utf-8") as f:
        f.write(json.dumps(log, ensure_ascii=False) + "\n")

    return reponse, log

# Démo
reponse, log = rag_avec_logging("Quel budget pour le Japon ?")
print("Réponse :", reponse[:80], "…\n")
print("Log enregistré :")
print(json.dumps(log, ensure_ascii=False, indent=2)[:400])

---

## Partie D — Mise en production (production-ready)

Passer d'un notebook qui marche à un service **fiable** utilisé par de vraies personnes. Les points essentiels :

### 🛡️ Fiabilité

| Sujet | Bonne pratique |
|---|---|
| **Rate limits / erreurs API** | Retry automatique + backoff (LangChain le fait déjà), gérer les `try/except` |
| **Fallback** | Si le LLM est indisponible, message clair plutôt que crash |
| **Cache** | Mettre en cache les questions fréquentes → moins de coût, plus rapide |
| **Timeout** | Ne pas laisser une requête pendre indéfiniment |
| **Garde-fou** | Si aucun chunk pertinent (score trop bas) → « je n'ai pas l'info » sans appeler le LLM |

### 💰 Coût

- **Embeddings en local** (comme ici avec FastEmbed) → 0€ sur la partie recherche
- **Cache** des réponses fréquentes → moins d'appels LLM
- **Choisir le bon modèle** : un petit modèle (flash) suffit souvent, inutile de payer un gros modèle
- **Suivre les tokens** consommés (loggés) pour éviter les mauvaises surprises

### 🔒 Sécurité

| Risque | Parade |
|---|---|
| Clé API exposée | Toujours dans `.env`, jamais dans le code ni sur GitHub (`.gitignore`) |
| Données sensibles (PII) | Filtrer/anonymiser avant d'envoyer au LLM |
| **Prompt injection** | Un document malveillant peut contenir « ignore tes instructions ». Valider les sources, cloisonner le contexte |
| Fuite de données | Vérifier la politique de rétention du fournisseur LLM |

### 📈 Observabilité

- **LangSmith** (par LangChain) : trace chaque appel, mesure latence/coût, rejoue les cas d'échec
- Alternatives : Langfuse, Phoenix (Arize), ou logs maison (comme notre `.jsonl`)
- **Alerting** : si Hit Rate < seuil ou latence > seuil → notification

### ✅ Checklist avant de déployer

- [ ] Golden dataset créé et RAG évalué (Hit Rate, MRR, groundedness)
- [ ] Garde-fou anti-hallucination (prompt strict + seuil de similarité)
- [ ] Clés API sécurisées (`.env` + `.gitignore`)
- [ ] Logging en place (questions, sources, latence, feedback)
- [ ] Gestion des erreurs (retry, fallback, timeout)
- [ ] Stratégie de mise à jour des documents (data drift)
- [ ] Monitoring + alertes configurés
- [ ] Coût estimé et suivi

---

## Récapitulatif

### Ce qu'on a mesuré

```
┌─────────────────────────────────────────────────────────────┐
│  ÉVALUATION OFFLINE (avant déploiement)                      │
│                                                             │
│  Partie A — RECHERCHE          Partie B — GÉNÉRATION         │
│  • Hit Rate @ k                • Groundedness (anti-hallu)   │
│  • MRR                         • Pertinence (answer rel.)    │
│  (local, gratuit)              (LLM-as-a-judge)              │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│  APRÈS DÉPLOIEMENT (en continu)                             │
│                                                             │
│  Partie C — MONITORING         Partie D — PRODUCTION         │
│  • Logging (question, latence) • Fiabilité (retry, cache)   │
│  • Détection de dérive         • Sécurité (clés, injection) │
│  • Boucle d'amélioration       • Observabilité (LangSmith)  │
└─────────────────────────────────────────────────────────────┘
```

### Les métriques essentielles à retenir

| Métrique | Étage | Mesure | Sans LLM ? |
|---|---|---|---|
| **Hit Rate @ k** | Recherche | Le bon doc est-il trouvé ? | ✅ oui |
| **MRR** | Recherche | Est-il bien classé ? | ✅ oui |
| **Groundedness** | Génération | Pas d'hallucination ? | ❌ (LLM-juge) |
| **Answer Relevance** | Génération | Répond à la question ? | ❌ (LLM-juge) |

### À retenir
- **On n'améliore que ce qu'on mesure** — un RAG sans évaluation est un RAG qu'on ne maîtrise pas
- **Évaluer les 2 étages séparément** : recherche d'abord (gratuit), génération ensuite
- Le **golden dataset** est la fondation de tout
- La **groundedness** est la métrique reine (elle attrape les hallucinations)
- En prod : **logger, surveiller la dérive, boucler** — un RAG se maintient dans le temps
- Outils pro : **RAGAS** (métriques RAG clés-en-main), **LangSmith** (observabilité)

### Pour aller plus loin
- **RAGAS** : librairie Python dédiée à l'évaluation RAG (faithfulness, context precision/recall…)
- **LangSmith** : plateforme de tracing et d'évaluation de LangChain
- Enrichir le golden dataset avec de **vraies** questions utilisateurs